In [ ]:
import base64
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

%matplotlib inline

# Giao diện chung cho mọi biểu đồ: nền sáng, lưới nhạt, tiêu đề đậm
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "axes.titleweight": "bold",
    "axes.titlesize": 13,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 100,
    "axes.edgecolor": "#cbd5e1",
    "grid.color": "#e2e8f0",
    "figure.facecolor": "white",
    "axes.titlelocation": "left",
})

# Bảng màu dùng thống nhất trong toàn bộ dashboard
PRIMARY, UP, DOWN, ACCENT = "#2563eb", "#16a34a", "#dc2626", "#f59e0b"

In [ ]:
FILE_PATH = "abc-20.csv"

# header=None: file không có dòng tiêu đề
raw = pd.read_csv(FILE_PATH, header=None)
raw.columns = [f"Giá {i}" for i in range(1, raw.shape[1] + 1)]
raw.index = np.arange(1, len(raw) + 1)      # trục thời gian = số thứ tự phiên (1..365)
raw.index.name = "Phiên"

# df = 7 cột gốc + chuỗi trung bình (chuỗi đại diện)
df = raw.copy()
df["Trung bình 7 cột"] = raw.mean(axis=1)

SERIES = list(df.columns)       # danh sách chuỗi để đưa vào menu chọn
N = len(df)

In [ ]:
def lay_du_lieu(ten_cot, p_dau, p_cuoi, ma_ngan, ma_dai, cua_so_bd):
    """Tính các chỉ báo cho một chuỗi giá rồi cắt theo khoảng phiên [p_dau, p_cuoi]."""
    gia = df[ten_cot]
    out = pd.DataFrame({"Giá": gia})

    out["Lợi suất %"] = gia.pct_change() * 100                      # % thay đổi so với phiên trước
    out["MA ngắn"] = gia.rolling(ma_ngan).mean()                    # trung bình trượt ngắn hạn
    out["MA dài"] = gia.rolling(ma_dai).mean()                      # trung bình trượt dài hạn
    do_lech = gia.rolling(ma_ngan).std()                            # Bollinger = MA ± 2 độ lệch chuẩn
    out["BB trên"] = out["MA ngắn"] + 2 * do_lech
    out["BB dưới"] = out["MA ngắn"] - 2 * do_lech
    out["Biến động trượt %"] = out["Lợi suất %"].rolling(cua_so_bd).std()

    sub = out.loc[p_dau:p_cuoi].copy()
    # Drawdown = mức giảm so với đỉnh gần nhất TRONG khoảng đang xem
    sub["Drawdown %"] = (sub["Giá"] / sub["Giá"].cummax() - 1) * 100
    return sub


def thong_ke_tom_tat(sub):
    """Bảng thống kê tóm tắt của khoảng đã lọc (trả về DataFrame 1 cột để hiển thị)."""
    gia, ls = sub["Giá"], sub["Lợi suất %"].dropna()
    q1, q2, q3 = gia.quantile([0.25, 0.5, 0.75])
    dong = [
        ("Số phiên", f"{len(gia):,}"),
        ("Giá đầu kỳ", f"{gia.iloc[0]:,.2f}"),
        ("Giá cuối kỳ", f"{gia.iloc[-1]:,.2f}"),
        ("Thay đổi cả kỳ", f"{(gia.iloc[-1] / gia.iloc[0] - 1) * 100:+.2f} %"),
        ("Giá trung bình", f"{gia.mean():,.2f}"),
        ("Độ lệch chuẩn giá", f"{gia.std():,.2f}"),
        ("Giá thấp nhất", f"{gia.min():,.2f}  (phiên {gia.idxmin()})"),
        ("Giá cao nhất", f"{gia.max():,.2f}  (phiên {gia.idxmax()})"),
        ("Q1 / Trung vị / Q3", f"{q1:,.0f} / {q2:,.0f} / {q3:,.0f}"),
        ("Lợi suất TB mỗi phiên", f"{ls.mean():+.3f} %"),
        ("Độ lệch chuẩn lợi suất", f"{ls.std():.3f} %"),
        ("Độ lệch (skewness)", f"{ls.skew():+.3f}"),
        ("Độ nhọn (excess kurtosis)", f"{ls.kurt():+.3f}"),
        ("VaR 5% (1 phiên)", f"{np.percentile(ls, 5):.2f} %"),
        ("Drawdown tối đa", f"{sub['Drawdown %'].min():.2f} %"),
    ]
    return pd.DataFrame(dong, columns=["Chỉ tiêu", "Giá trị"]).set_index("Chỉ tiêu")


def bang_dong_cuoi(sub, n_dong, ma_ngan, ma_dai, hien_bb):
    """Bảng n dòng cuối của khoảng đã lọc."""
    cot = ["Giá", "Lợi suất %", "MA ngắn", "MA dài"] + (["BB trên", "BB dưới"] if hien_bb else [])
    t = sub[cot].tail(n_dong).rename(columns={"MA ngắn": f"MA{ma_ngan}", "MA dài": f"MA{ma_dai}"})
    return t

In [ ]:
def ve_bieu_do_gia(sub, ten, loai, hien_ma, hien_bb, ma_ngan, ma_dai):
    """Biểu đồ 1: giá + MA + Bollinger (3 kiểu: Đường / Diện tích / Cột)."""
    x, gia = sub.index, sub["Giá"]
    fig, ax = plt.subplots(figsize=(11, 5.2))
    handles = []

    if loai == "Cột":
        # Cột xanh = phiên tăng, đỏ = phiên giảm so với phiên trước
        mau = np.where(sub["Lợi suất %"].fillna(0) >= 0, UP, DOWN)
        ax.bar(x, gia, color=mau, width=0.9, alpha=0.75)
        handles += [plt.Rectangle((0, 0), 1, 1, color=UP, alpha=0.75, label="Phiên tăng"),
                    plt.Rectangle((0, 0), 1, 1, color=DOWN, alpha=0.75, label="Phiên giảm")]
    else:
        ax.plot(x, gia, color=PRIMARY, lw=1.8, label=f"Giá ({ten})")
        if loai == "Diện tích":
            ax.fill_between(x, gia, gia.min(), color=PRIMARY, alpha=0.15)

    if hien_bb:       # dải Bollinger: vùng mờ giữa BB trên và BB dưới
        ax.fill_between(x, sub["BB dưới"], sub["BB trên"], color="gray", alpha=0.15, label="Bollinger (±2σ)")
    if hien_ma:
        ax.plot(x, sub["MA ngắn"], color=ACCENT, lw=1.8, label=f"MA{ma_ngan}")
        ax.plot(x, sub["MA dài"], color="#7c3aed", lw=1.8, label=f"MA{ma_dai}")

    # Đánh dấu đỉnh và đáy của khoảng đang xem
    i_max, i_min = gia.idxmax(), gia.idxmin()
    ax.scatter([i_max], [gia[i_max]], color=UP, s=70, zorder=5, edgecolor="white")
    ax.scatter([i_min], [gia[i_min]], color=DOWN, s=70, zorder=5, edgecolor="white")
    def dat_nhan(i, nhan, mau, dy):
        # Điểm nằm nửa phải biểu đồ thì đặt chữ lệch sang trái để không bị cắt mép
        ben_phai = (i - x[0]) / max(x[-1] - x[0], 1) > 0.6
        ax.annotate(nhan, (i, gia[i]), xytext=(-8 if ben_phai else 8, dy), textcoords="offset points",
                    ha="right" if ben_phai else "left", fontsize=9, color=mau, fontweight="bold",
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.85))
    dat_nhan(i_max, f"Cao nhất {gia[i_max]:,.0f} (phiên {i_max})", UP, 8)
    dat_nhan(i_min, f"Thấp nhất {gia[i_min]:,.0f} (phiên {i_min})", DOWN, -16)

    h, _ = ax.get_legend_handles_labels()
    ax.legend(handles=h + handles, loc="upper center", bbox_to_anchor=(0.5, -0.12),
              ncol=6, frameon=False, fontsize=9)
    ax.set(title=f"{ten} — phiên {x[0]} đến {x[-1]}", xlabel="Số thứ tự phiên", ylabel="Giá")
    ax.set_xlim(x[0], x[-1])
    fig.tight_layout()
    return fig


def ve_so_sanh_7_cot(p_dau, p_cuoi):
    """Biểu đồ 2: so sánh 7 cột giá + ma trận tương quan lợi suất."""
    seg = raw.loc[p_dau:p_cuoi]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 5.2), gridspec_kw={"width_ratios": [1.6, 1]})

    for mau, cot in zip(sns.color_palette("tab10", 7), seg.columns):
        ax1.plot(seg.index, seg[cot], color=mau, lw=1, alpha=0.6, label=cot)
    ax1.plot(seg.index, seg.mean(axis=1), color="black", lw=2.5, label="Trung bình")
    ax1.set(title="Giá của 7 cột và đường trung bình", xlabel="Số thứ tự phiên", ylabel="Giá")
    ax1.legend(ncol=4, fontsize=8, loc="upper left")
    ax1.set_xlim(seg.index[0], seg.index[-1])

    # Tương quan của LỢI SUẤT (không phải giá) mới phản ánh đúng mức "đi cùng nhau"
    ls = raw.pct_change().loc[p_dau:p_cuoi]
    corr = ls.corr()
    tb_corr = corr.values[np.triu_indices(7, 1)].mean()
    corr.index = corr.columns = [f"G{i}" for i in range(1, 8)]
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, center=0,
                square=True, linewidths=0.5, cbar_kws={"shrink": 0.8}, ax=ax2)
    ax2.set_title(f"Tương quan lợi suất (TB = {tb_corr:.2f})")
    fig.tight_layout()
    return fig


def ve_phan_phoi_loi_suat(sub, ten):
    """Biểu đồ 3: histogram + KDE + đường chuẩn lý thuyết, kèm Q-Q plot."""
    ls = sub["Lợi suất %"].dropna()
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 5), gridspec_kw={"width_ratios": [1.5, 1]})

    sns.histplot(ls, bins=30, stat="density", color=PRIMARY, alpha=0.35, ax=ax1, label="Lợi suất thực tế")
    sns.kdeplot(ls, color=PRIMARY, lw=2, ax=ax1, label="KDE")
    xs = np.linspace(ls.min(), ls.max(), 300)
    ax1.plot(xs, stats.norm.pdf(xs, ls.mean(), ls.std()), color=ACCENT, ls="--", lw=2, label="Phân phối chuẩn")
    var5 = np.percentile(ls, 5)
    ax1.axvline(var5, color=DOWN, ls=":", lw=2, label=f"VaR 5% = {var5:.1f}%")
    ax1.axvline(ls.mean(), color="black", ls="-", lw=1, label=f"Trung bình = {ls.mean():.2f}%")

    p_sw = stats.shapiro(ls).pvalue
    ghi_chu = (f"n = {len(ls)}\nĐộ lệch chuẩn = {ls.std():.2f}%\nSkewness = {ls.skew():+.2f}\n"
               f"Kurtosis = {ls.kurt():+.2f}\nShapiro p = {p_sw:.3g}")
    ax1.text(0.98, 0.97, ghi_chu, transform=ax1.transAxes, ha="right", va="top", fontsize=9,
             bbox=dict(boxstyle="round", facecolor="white", edgecolor="#cbd5e1"))
    ax1.set(title=f"Phân phối lợi suất mỗi phiên — {ten}", xlabel="Lợi suất (%)", ylabel="Mật độ")
    ax1.legend(loc="upper left", fontsize=9)

    # Q-Q plot: các điểm càng bám sát đường thẳng thì càng gần phân phối chuẩn
    stats.probplot(ls, dist="norm", plot=ax2)
    pts, line = ax2.get_lines()[0], ax2.get_lines()[1]
    pts.set(markerfacecolor=PRIMARY, markeredgecolor=PRIMARY, markersize=4, alpha=0.6)
    line.set(color=ACCENT, lw=2)
    ax2.set(title="Q-Q plot so với phân phối chuẩn", xlabel="Phân vị lý thuyết", ylabel="Phân vị mẫu")
    fig.tight_layout()
    return fig


def ve_bien_dong_truot(sub, ten, cua_so_bd):
    """Biểu đồ 4: độ biến động trượt (độ lệch chuẩn lợi suất) và drawdown."""
    x, bd = sub.index, sub["Biến động trượt %"]
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6.4), sharex=True,
                                   gridspec_kw={"height_ratios": [1.4, 1]})

    ax1.plot(x, bd, color=PRIMARY, lw=1.8)
    ax1.fill_between(x, bd, color=PRIMARY, alpha=0.15)
    ax1.axhline(bd.mean(), color=ACCENT, ls="--", lw=1.5, label=f"Trung bình = {bd.mean():.2f}%")
    i_cao = bd.idxmax()
    ax1.scatter([i_cao], [bd[i_cao]], color=DOWN, s=60, zorder=5, edgecolor="white")
    ax1.annotate(f"Biến động cao nhất {bd[i_cao]:.2f}% (phiên {i_cao})", (i_cao, bd[i_cao]),
                 xytext=(8, 6), textcoords="offset points", fontsize=9, color=DOWN, fontweight="bold")
    ax1.set(title=f"Biến động trượt {cua_so_bd} phiên — {ten}", ylabel="Độ lệch chuẩn lợi suất (%)")
    ax1.legend(loc="upper left", fontsize=9)

    ax2.fill_between(x, sub["Drawdown %"], 0, color=DOWN, alpha=0.35)
    ax2.plot(x, sub["Drawdown %"], color=DOWN, lw=1.2)
    ax2.set(title="Drawdown (mức giảm so với đỉnh gần nhất)", xlabel="Số thứ tự phiên", ylabel="Drawdown (%)")
    ax2.set_xlim(x[0], x[-1])
    fig.tight_layout()
    return fig

In [ ]:
KIEU = {"description_width": "115px"}
RONG = widgets.Layout(width="100%")

# ---- Nhóm lọc dữ liệu
w_chuoi = widgets.Dropdown(options=SERIES, value="Trung bình 7 cột",
                           description="Chuỗi giá:", style=KIEU, layout=RONG)
w_khoang = widgets.IntRangeSlider(value=[1, N], min=1, max=N, step=1, description="Khoảng phiên:",
                                  continuous_update=False, style=KIEU, layout=RONG)
w_loai = widgets.ToggleButtons(options=["Đường", "Diện tích", "Cột"], value="Đường",
                               description="Loại biểu đồ:", style={"description_width": "115px", "button_width": "66px"})

# ---- Nhóm chỉ báo kỹ thuật
w_ma = widgets.Checkbox(value=True, description="Hiện đường MA", indent=False)
w_ma_ngan = widgets.IntSlider(value=20, min=5, max=60, description="MA ngắn (phiên):", style=KIEU, layout=RONG)
w_ma_dai = widgets.IntSlider(value=50, min=20, max=120, description="MA dài (phiên):", style=KIEU, layout=RONG)
w_bb = widgets.Checkbox(value=False, description="Hiện Bollinger (MA ngắn ± 2σ)", indent=False)
w_bd = widgets.IntSlider(value=20, min=5, max=60, description="Cửa sổ biến động:", style=KIEU, layout=RONG)

# ---- Nút điều khiển
btn_cap_nhat = widgets.Button(description="Cập nhật", button_style="primary", icon="refresh")
btn_dat_lai = widgets.Button(description="Đặt lại", button_style="warning", icon="undo")
btn_xuat = widgets.Button(description="Xuất CSV", button_style="success", icon="download")

# Giá trị mặc định (dùng cho nút Đặt lại)
MAC_DINH = [(w_chuoi, "Trung bình 7 cột"), (w_khoang, (1, N)), (w_loai, "Đường"),
            (w_ma, True), (w_ma_ngan, 20), (w_ma_dai, 50), (w_bb, False), (w_bd, 20)]

# Kiểu dáng nút: "Cập nhật" chiếm cả hàng, hai nút còn lại chia đôi
btn_cap_nhat.layout = widgets.Layout(width="100%", height="42px", margin="12px 0 0 0")
for b in (btn_dat_lai, btn_xuat):
    b.layout = widgets.Layout(width="49%", height="36px")

In [ ]:
# ===== GIAO DIỆN (CSS) =====
CSS = """
<style>
.abc-root{font-family:Inter,'Segoe UI',system-ui,-apple-system,sans-serif;color:#0f172a}
.abc-banner{position:relative;overflow:hidden;background:linear-gradient(120deg,#0f172a 0%,#1e3a8a 55%,#2563eb 100%);color:#fff;padding:26px 30px;border-radius:18px;margin-bottom:8px;box-shadow:0 10px 30px rgba(30,58,138,.25)}
.abc-banner:after{content:"";position:absolute;right:-60px;top:-60px;width:240px;height:240px;border-radius:50%;background:radial-gradient(circle,rgba(56,189,248,.45),transparent 70%)}
.abc-banner h1{margin:0;font-size:26px;font-weight:800;letter-spacing:-.3px;color:#fff}
.abc-banner p{margin:8px 0 12px;font-size:14px;color:#cbd5e1}
.abc-chip{display:inline-block;background:rgba(255,255,255,.14);border:1px solid rgba(255,255,255,.25);color:#fff;padding:3px 12px;border-radius:999px;font-size:12px;margin-right:8px}
.abc-sec{display:flex;align-items:center;gap:10px;font-size:17px;font-weight:700;color:#0f172a;margin:26px 0 10px}
.abc-sec .n{display:inline-flex;align-items:center;justify-content:center;width:28px;height:28px;border-radius:8px;background:#2563eb;color:#fff;font-size:14px}
.abc-card{background:#fff;border:1px solid #e2e8f0;border-radius:16px;padding:14px 18px;box-shadow:0 2px 8px rgba(15,23,42,.06)}
.abc-card-title{font-weight:700;color:#1e3a8a;font-size:14px;margin-bottom:8px}
.abc-kpis{display:grid;grid-template-columns:repeat(auto-fit,minmax(170px,1fr));gap:12px;margin:4px 0 6px}
.abc-kpi{position:relative;background:#fff;border:1px solid #e2e8f0;border-radius:16px;padding:12px 16px 8px;box-shadow:0 2px 8px rgba(15,23,42,.06);transition:transform .15s,box-shadow .15s}
.abc-kpi:hover{transform:translateY(-2px);box-shadow:0 8px 20px rgba(15,23,42,.12)}
.abc-kpi:before{content:"";position:absolute;left:0;top:14px;bottom:14px;width:4px;border-radius:0 4px 4px 0;background:var(--c)}
.abc-kpi .t{font-size:12px;color:#64748b;font-weight:600;text-transform:uppercase;letter-spacing:.4px}
.abc-kpi .v{font-size:24px;font-weight:800;color:var(--c);line-height:1.25}
.abc-kpi svg{display:block;width:100%;height:26px;margin-top:2px}
.abc-table{border-collapse:collapse;font-size:13px;width:100%;color:#0f172a}
.abc-table td,.abc-table th{padding:7px 12px;border-bottom:1px solid #eef2f7;text-align:right}
.abc-table td:first-child,.abc-table th:first-child{text-align:left;color:#475569}
.abc-table tbody tr:hover{background:#f8fafc}
.abc-table thead th{background:#eff6ff;color:#1e3a8a;font-weight:700;border-bottom:2px solid #bfdbfe}
body{background:#f4f7fb}
.abc-root img{max-width:100%;height:auto}
.abc-banner{display:flex;align-items:center;justify-content:space-between;flex-wrap:wrap;gap:8px}
.abc-group{font-size:13px;font-weight:800;color:#1e3a8a;text-transform:uppercase;letter-spacing:.5px;margin:14px 0 6px;padding-bottom:5px;border-bottom:2px solid #dbeafe}
.abc-group:first-child{margin-top:2px}
.abc-hint{font-size:12px;color:#64748b;margin-top:8px}
.abc-note{padding:10px 16px;border-radius:12px;font-size:13px;margin:8px 0;border:1px solid rgba(0,0,0,.05)}
.abc-sub{font-size:14px;font-weight:700;color:#334155;margin:12px 0 8px}
.abc-root .jupyter-button{border-radius:12px;font-weight:700;box-shadow:0 2px 6px rgba(15,23,42,.15);transition:transform .12s,filter .12s}
.abc-root .jupyter-button:hover{transform:translateY(-1px);filter:brightness(1.07)}
.abc-root .ui-slider .ui-slider-handle,.abc-root .noUi-handle{background:#2563eb;border-color:#fff;box-shadow:0 0 0 2px #2563eb}
.abc-root .ui-slider-range{background:#93c5fd}
.abc-root .lm-TabBar-tab,.abc-root .p-TabBar-tab{font-weight:600;border-radius:10px 10px 0 0}
.abc-root .lm-TabBar-tab.lm-mod-current,.abc-root .p-TabBar-tab.p-mod-current{color:#1e3a8a;border-top:3px solid #2563eb;background:#fff}
</style>
"""

def spark(y, mau, n=40):
    """Biểu đồ mini (SVG) cho thẻ KPI."""
    y = pd.Series(y).dropna()
    y = y.iloc[np.linspace(0, len(y) - 1, min(n, len(y))).astype(int)].to_numpy()
    lo, hi = y.min(), y.max()
    z = (y - lo) / (hi - lo) if hi > lo else np.full(len(y), .5)
    pts = " ".join(f"{i / (len(z) - 1) * 100:.1f},{24 - v * 22:.1f}" for i, v in enumerate(z))
    return (f"<svg viewBox='0 0 100 26' preserveAspectRatio='none'>"
            f"<polyline points='{pts}' fill='none' stroke='{mau}' stroke-width='1.8' vector-effect='non-scaling-stroke'/></svg>")

def html_kpi(sub):
    """6 thẻ chỉ số nhanh của khoảng đang xem (kèm biểu đồ mini)."""
    gia, ls = sub["Giá"], sub["Lợi suất %"].dropna()
    dt = (gia.iloc[-1] / gia.iloc[0] - 1) * 100
    the = [("Giá cuối kỳ", f"{gia.iloc[-1]:,.0f}", PRIMARY, gia),
           ("Thay đổi cả kỳ", f"{dt:+.1f}%", UP if dt >= 0 else DOWN, gia / gia.iloc[0]),
           ("Giá cao nhất", f"{gia.max():,.0f}", UP, gia.cummax()),
           ("Giá thấp nhất", f"{gia.min():,.0f}", DOWN, gia.cummin()),
           ("Biến động / phiên", f"{ls.std():.2f}%", ACCENT, sub["Biến động trượt %"]),
           ("Drawdown tối đa", f"{sub['Drawdown %'].min():.1f}%", DOWN, sub["Drawdown %"])]
    return "<div class='abc-kpis'>" + "".join(
        f"<div class='abc-kpi' style='--c:{m}'><div class='t'>{t}</div><div class='v'>{v}</div>{spark(y, m)}</div>"
        for t, v, m, y in the) + "</div>"

def html_chi_tieu(tk):
    """Bảng thống kê tóm tắt chia 2 cột cho gọn."""
    dong = [f"<tr><td>{k}</td><td>{v}</td></tr>" for k, v in tk["Giá trị"].items()]
    nua = (len(dong) + 1) // 2
    bang = lambda r: f"<table class='abc-table'>{''.join(r)}</table>"
    return (f"<div style='display:flex;gap:18px;flex-wrap:wrap'><div style='flex:1;min-width:300px'>{bang(dong[:nua])}</div>"
            f"<div style='flex:1;min-width:300px'>{bang(dong[nua:])}</div></div>")

def thong_bao(loai, noi_dung):
    mau = {"loi": ("#fef2f2", "#991b1b"), "ok": ("#f0fdf4", "#166534")}[loai]
    return HTML(f"<div class='abc-note' style='background:{mau[0]};color:{mau[1]}'>{noi_dung}</div>")

def the_khung(tieu_de, *con):
    """Khung (card) chứa một nhóm widget."""
    box = widgets.VBox([widgets.HTML(f"<div class='abc-card-title'>{tieu_de}</div>"), *con],
                       layout=widgets.Layout(width="49%", min_width="340px", margin="0 0 10px 0"))
    box.add_class("abc-card")
    return box

def muc(so, txt):
    return widgets.HTML(f"<div class='abc-sec'><span class='n'>{so}</span>{txt}</div>")

In [ ]:
# ===== LOGIC NÚT BẤM =====
out_thong_bao, out_kpi, out_bang = widgets.Output(), widgets.Output(), widgets.Output()
TEN_TAB = ["Giá + MA", "So sánh 7 cột", "Phân phối lợi suất", "Biến động trượt", "Bảng số liệu"]
out_tabs = [widgets.Output() for _ in range(4)]
tab = widgets.Tab(children=out_tabs + [out_bang])    # tab thứ 5 = bảng số liệu
for i, t in enumerate(TEN_TAB):
    tab.set_title(i, t)

TRANG_THAI = {"sub": None, "p_dau": None, "p_cuoi": None}    # nhớ dữ liệu lần cập nhật gần nhất


def cap_nhat(_=None):
    """Kiểm tra tham số → tính chỉ báo → cập nhật thẻ KPI, bảng và 4 biểu đồ."""
    ten, (p_dau, p_cuoi) = w_chuoi.value, w_khoang.value
    ma_n, ma_d, cs_bd = w_ma_ngan.value, w_ma_dai.value, w_bd.value
    n_dong = 5          # luôn hiển thị 5 phiên cuối

    with out_thong_bao:
        clear_output(wait=True)
        if ma_d <= ma_n:
            display(thong_bao("loi", "⚠️ MA dài phải lớn hơn MA ngắn. Hãy chỉnh lại rồi bấm <b>Cập nhật</b>."))
            return
        if p_cuoi - p_dau < 10:
            display(thong_bao("loi", "⚠️ Hãy chọn khoảng ít nhất 10 phiên để thống kê có ý nghĩa."))
            return

    sub = lay_du_lieu(ten, p_dau, p_cuoi, ma_n, ma_d, cs_bd)
    TRANG_THAI.update(sub=sub, p_dau=p_dau, p_cuoi=p_cuoi)

    with out_kpi:                                    # thẻ chỉ số nhanh
        clear_output(wait=True)
        display(HTML(f"<div class='abc-sub'>{ten} · phiên {p_dau} → {p_cuoi}</div>" + html_kpi(sub)))

    with out_bang:                                   # bảng số liệu
        clear_output(wait=True)
        display(HTML("<div class='abc-sub'>Thống kê tóm tắt</div>" + html_chi_tieu(thong_ke_tom_tat(sub))))
        display(HTML(f"<div class='abc-sub'>{n_dong} phiên cuối của khoảng đã lọc</div>"))
        bang = bang_dong_cuoi(sub, n_dong, ma_n, ma_d, w_bb.value)
        try:    # tô màu lợi suất tăng (xanh) / giảm (đỏ)
            sty = bang.style.format("{:,.2f}", na_rep="–").set_table_attributes("class='abc-table'")
            tomau = sty.map if hasattr(sty, "map") else sty.applymap
            sty = tomau(lambda v: f"color: {UP}" if v > 0 else (f"color: {DOWN}" if v < 0 else ""), subset=["Lợi suất %"])
            display(HTML(sty.to_html()))
        except Exception:
            display(bang.round(2))

    ve = [lambda: ve_bieu_do_gia(sub, ten, w_loai.value, w_ma.value, w_bb.value, ma_n, ma_d),   # 4 biểu đồ
          lambda: ve_so_sanh_7_cot(p_dau, p_cuoi),
          lambda: ve_phan_phoi_loi_suat(sub, ten),
          lambda: ve_bien_dong_truot(sub, ten, cs_bd)]
    for out, ham in zip(out_tabs, ve):
        with out:
            clear_output(wait=True)
            fig = ham()
            plt.show()
            plt.close(fig)      # giải phóng bộ nhớ trên server


def dat_lai(_=None):
    for w, v in MAC_DINH:
        w.value = v
    out_thong_bao.clear_output()
    cap_nhat()


def xuat_csv(_=None):
    """Tạo link tải CSV nhúng sẵn dữ liệu (base64) → dùng được cả trên web Voilà/Render."""
    if TRANG_THAI["sub"] is None:
        cap_nhat()
    if TRANG_THAI["sub"] is None:       # cập nhật thất bại do tham số không hợp lệ
        return
    sub = TRANG_THAI["sub"]
    ten_file = f"abc20_loc_{TRANG_THAI['p_dau']}_{TRANG_THAI['p_cuoi']}.csv"
    b64 = base64.b64encode(sub.round(4).to_csv().encode("utf-8-sig")).decode()
    link = (f'<a download="{ten_file}" href="data:text/csv;base64,{b64}" style="font-weight:700;color:#166534">'
            f'⬇️ Bấm để tải {ten_file}</a> ({len(sub)} dòng)')
    with out_thong_bao:
        clear_output(wait=True)
        display(thong_bao("ok", "✅ Đã chuẩn bị dữ liệu đã lọc. " + link))


btn_cap_nhat.on_click(cap_nhat)
btn_dat_lai.on_click(dat_lai)
btn_xuat.on_click(xuat_csv)

In [ ]:
# ===== BỐ CỤC: SIDEBAR (điều khiển) + KHU VỰC CHÍNH =====
def nhom(txt):
    return widgets.HTML(f"<div class='abc-group'>{txt}</div>")

banner = widgets.HTML(CSS + f"""
<div class='abc-banner'>
  <div>
    <h1>📈 Dashboard phân tích biến động giá cổ phiếu ABC-20</h1>
    <p>Chọn bộ lọc ở cột trái rồi bấm <b>Cập nhật</b> để xem thẻ chỉ số, biểu đồ và bảng số liệu.</p>
  </div>
  <div><span class='abc-chip'>{N} phiên</span><span class='abc-chip'>7 chuỗi + trung bình</span></div>
</div>""")

sidebar = widgets.VBox([
    nhom("🔎 Bộ lọc dữ liệu"), w_chuoi, w_khoang, w_loai,
    nhom("📐 Chỉ báo kỹ thuật"), w_ma, w_ma_ngan, w_ma_dai, w_bb, w_bd,
    btn_cap_nhat,
    widgets.HBox([btn_dat_lai, btn_xuat], layout=widgets.Layout(justify_content="space-between", margin="8px 0 0 0")),
    out_thong_bao,
    widgets.HTML("<div class='abc-hint'>Đổi bộ lọc xong, bấm <b>Cập nhật</b> để vẽ lại.</div>"),
], layout=widgets.Layout(width="340px", flex="0 0 340px", margin="0 16px 16px 0"))
sidebar.add_class("abc-card")

khu_chinh = widgets.VBox([out_kpi, tab], layout=widgets.Layout(flex="1 1 560px", min_width="0"))

giao_dien = widgets.VBox([
    banner,
    widgets.HBox([sidebar, khu_chinh], layout=widgets.Layout(width="100%", align_items="flex-start", flex_flow="row wrap")),
], layout=widgets.Layout(width="100%", max_width="1400px"))
giao_dien.add_class("abc-root")

display(giao_dien)
cap_nhat()      # vẽ lần đầu với tham số mặc định